# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples using mass spectrometry. Multiple record sets and fields are described in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}\nIdentifier: {metadata.identifier}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all available record sets, their @id, and contained fields

record_sets = []
# Collect croissant IDs for all record sets
if hasattr(metadata, 'record_sets'):
    # mlcroissant >=0.6.0 uses .record_sets
    for rs in metadata.record_sets:
        print(f"RecordSet: {rs['@id']} (name: {rs.get('name', None)})")
        if 'fields' in rs:
            print("  Fields:")
            for fld in rs['fields']:
                if isinstance(fld, dict):
                    print(f"    - {fld.get('@id', '')} (name: {fld.get('name', None)})")
                else:
                    print(f"    - {fld}")
        record_sets.append(rs['@id'])
        print()
else:
    # Some schema may use .record_set as a list of dicts
    try:
        rs_list = getattr(metadata, 'record_set', getattr(metadata, 'recordSets', []))
        for rs in rs_list:
            print(f"RecordSet: {rs['@id']} (name: {rs.get('name', None)})")
            if 'field' in rs:
                print("  Fields:")
                for fld in rs['field']:
                    if isinstance(fld, dict):
                        print(f"    - {fld.get('@id', '')} (name: {fld.get('name', None)})")
                    else:
                        print(f"    - {fld}")
            record_sets.append(rs['@id'])
            print()
    except Exception as exc:
        print("Record set structure not found in metadata.")
        print("Error:", exc)

# If no record sets found, attempt to infer via mlcroissant API, as fallback
if not record_sets:
    # This fallback assumes mlcroissant exposes record_set info via .record_sets property
    try:
        for rs in dataset.record_sets:
            print(f"RecordSet: {rs['@id']} (name: {rs.get('name', None)})")
            record_sets.append(rs['@id'])
    except Exception as exc:
        print("Unable to retrieve record set IDs:", exc)

if record_sets:
    print("Available record sets (by @id):", record_sets)
else:
    print("No record sets found in the metadata. Please check the dataset schema.")

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Manually specify record set(s) by their @id found in the overview above.
# Example for FAIR^2: Try to auto-detect record_sets if not provided in metadata.

from mlcroissant._dataset import croissant

if not record_sets:
    # If previous cell found none, attempt to parse record_sets from croissant file
    croissant_obj = croissant.Croissant(croissant_url)
    if hasattr(croissant_obj, 'record_sets'):
        for rs in croissant_obj.record_sets:
            rs_id = rs.get('@id', None)
            if rs_id:
                record_sets.append(rs_id)

if not record_sets:
    raise RuntimeError("No record sets detected in the Croissant schema.")

# Print which record set(s) will be extracted
print(f"Extracting the following record sets: {record_sets}")

dataframes = {}
for record_set in record_sets:
    try:
        # Load up to 5k records for preview, modify for full load.
        records = list(dataset.records(record_set=record_set, limit=5000))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records for: {record_set}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as exc:
        print(f"Could not load records for record_set {record_set}: {exc}")

# Preview first record set
if dataframes:
    main_record_set = record_sets[0]
    print(f"Showing preview for {main_record_set}:")
    display(dataframes[main_record_set].head())
else:
    print("No dataframes extracted. Check if dataset has records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping. All fields should be referenced by their `@id` (column label).

In [ ]:
# Please select a record set and numeric field @id for analysis.
selected_record_set = main_record_set  # e.g. record_sets[0]

df = dataframes[selected_record_set]
print("Available columns in main record set:\n", df.columns.tolist())

# You may need to adjust field IDs below to match your schema columns
# For demonstration, select a likely numeric field:
suspected_numeric_fields = [col for col in df.columns if ('coverage' in col or 'MW' in col or 'count' in col or 'abundance' in col or 'score' in col or df[col].dtype.kind in 'fi')]
if not suspected_numeric_fields:
    suspected_numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if suspected_numeric_fields:
    numeric_field = suspected_numeric_fields[0]
else:
    raise Exception("No candidate numeric fields found in the main DataFrame.")
print(f"Selected numeric_field for EDA: {numeric_field}")

threshold = float(df[numeric_field].mean() + df[numeric_field].std() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0)
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optionally group by another field, e.g. a sample or protein description
candidate_group_fields = [col for col in df.columns if col not in [numeric_field] and df[col].dtype == object]
group_field = candidate_group_fields[0] if candidate_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field identified for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the main numeric field
plt.figure(figsize=(7, 4))
df[numeric_field].hist(bins=30, color='skyblue', edgecolor='k')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If a group field is detected, show a boxplot by group
if group_field and group_field in df.columns:
    top_groups = df[group_field].value_counts().head(10).index
    plt.figure(figsize=(12, 5))
    df_subset = df[df[group_field].isin(top_groups)]
    df_subset.boxplot(column=numeric_field, by=group_field, rot=45)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, analyze, and visualize the FAIR² mass spectrometry dataset using `mlcroissant` APIs. Key steps included listing available record sets and fields by `@id`, performing basic filtering and normalization, and visualizing numeric attributes. Adjust field references and thresholds to suit more advanced analyses or domain-specific questions as needed.